<div align="center">

  <a href="https://grayboxtech.github.io/weightslab/latest/index.html" target="_blank">
    <img width="100%" src="https://raw.githubusercontent.com/GrayboxTech/.github/main/profile/weightslab-banner-light.png" alt="WeightsLab banner"></a>

  <a href="https://github.com/GrayboxTech/weightslab/blob/main/LICENSE"><img src="https://img.shields.io/badge/License-Apache%202.0-blue.svg" alt="License"></a>
  <a href="https://github.com/GrayboxTech/weightslab/stargazers"><img src="https://img.shields.io/github/stars/GrayboxTech/weightslab?style=flat&color=5865F2" alt="Stars"></a>
  <a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/v/weightslab?style=flat&color=5865F2&logo=pypi&logoColor=white" alt="Version"></a>
  <br>
  <a href="https://colab.research.google.com/github/GrayboxTech/weightslab/blob/main/weightslab/examples/Notebooks/PyTorch/ws-classification.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open WeightsLab In Colab"></a>

  Welcome to the WeightsLab image-classification 🚀 notebook! <a href="https://github.com/GrayboxTech/weightslab">WeightsLab</a> is an open-source PyTorch tool for dataset debugging, mislabel detection, and mid-training data curation. We hope this notebook helps you get the most out of it — browse the <a href="https://grayboxtech.github.io/weightslab/latest/index.html">Docs</a> for details, and raise an issue on <a href="https://github.com/GrayboxTech/weightslab">GitHub</a> for support!</div>

# Image Classification with WeightsLab

This notebook trains a small CNN on **MNIST** and instruments it with WeightsLab so every training signal is traced **back to the exact samples** producing it.

The idea is simple: most data problems (mislabels, outliers, class imbalance) stay invisible until your model tells you through the loss. By wrapping your training objects with `wl.watch_or_edit(...)`, WeightsLab records **per-sample** loss and metrics live — so you can rank the worst samples, spot bad labels, and curate the dataset **without restarting training**.

### What you'll do
1. Install WeightsLab.
2. Wrap the model, optimizer, dataloaders, loss, and metric with the SDK.
3. Train for a few hundred steps while per-sample signals are captured.
4. Inspect the highest-loss samples right in the notebook — and see how to open the live **Weights Studio** UI.

## Setup

Install WeightsLab from PyPI. On Colab, the free **T4 GPU** runtime is plenty for this demo (`Runtime → Change runtime type → T4 GPU`).

In [ ]:
%pip install -q weightslab
!pip install --upgrade "protobuf>=6.31.1"
%pip install torchvision>=0.16

## 1. Imports

`weightslab` is imported as `wl`. The two `guard_*_context` managers scope a block as training vs. evaluation so signals are attributed to the right phase.

In [ ]:
import os
import tempfile
import logging

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset
from torchvision import datasets, transforms
from torchmetrics.classification import Accuracy
from tqdm.auto import tqdm

import weightslab as wl
from weightslab.examples.utils.baseline_models.pytorch.models import FashionCNN as CNN
from weightslab.components.global_monitoring import (
    guard_training_context,
    guard_testing_context,
)

logging.basicConfig(level=logging.ERROR)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. A dataset that carries sample identity

WeightsLab attributes every signal to a **sample id**. This thin wrapper around torchvision's MNIST returns `(image, id, label)` and attaches a virtual filepath per sample, so signals can later be traced back to individual images in the UI.

In [ ]:
class MNISTCustomDataset(Dataset):
    """MNIST that returns (image, index, label) and tracks a virtual filepath."""

    def __init__(self, root, train=True, download=False, transform=None, max_samples=None):
        self.mnist = datasets.MNIST(root=root, train=train, download=download, transform=None)
        self.transform = transform
        self.train = train
        self.max_samples = max_samples
        split = "train" if train else "test"
        self.filepaths = {}
        for idx in range(len(self.mnist)):
            if max_samples is not None and idx >= max_samples:
                break
            label = self.mnist.targets[idx].item()
            self.filepaths[idx] = os.path.join(
                "MNIST", "processed", split, f"class_{label}", f"sample_{idx:05d}.pt"
            )

    def __len__(self):
        if self.max_samples is not None:
            return min(len(self.mnist), self.max_samples)
        return len(self.mnist)

    def __getitem__(self, idx):
        image, label = self.mnist[idx]
        if self.transform:
            image = self.transform(image)
        return image, idx, label

## 3. Hyperparameters

We keep the run short so the notebook finishes quickly. Wrapping the config with `flag="hyperparameters"` lets the Studio UI read (and live-edit) these values while training.

In [ ]:
log_dir = tempfile.mkdtemp(prefix="weightslab_mnist_")

parameters = {
    "experiment_name": "mnist_classification",
    "device": str(device),
    "training_steps_to_do": 2000,         # raise for a longer live session
    "eval_full_to_train_steps_ratio": 100, # full eval every 100 steps
    "write_export_ratio": 100,             # export signals every 100 steps
    "root_log_dir": log_dir,
    "serving_grpc": True,                  # expose gRPC so Studio can connect
}

wl.watch_or_edit(parameters, flag="hyperparameters", poll_interval=1.0)
print("Experiment logs ->", log_dir)

## 4. Wrap the training objects

This is the heart of WeightsLab. Each object is passed through `wl.watch_or_edit(...)` with a `flag` describing its role. The returned objects behave exactly like the originals, but now report their state and per-sample signals to WeightsLab.

In [ ]:
data_root = os.path.join(log_dir, "data")
os.makedirs(data_root, exist_ok=True)

train_ds = MNISTCustomDataset(root=data_root, train=True, download=True,
                              transform=transforms.ToTensor(), max_samples=6000)
test_ds = MNISTCustomDataset(root=data_root, train=False, download=True,
                             transform=transforms.ToTensor(), max_samples=2000)

# Model + optimizer
model = wl.watch_or_edit(CNN().to(device), flag="model", device=device)
optimizer = wl.watch_or_edit(optim.Adam(model.parameters(), lr=0.01), flag="optimizer")

# Tracked dataloaders
train_loader = wl.watch_or_edit(
    train_ds, flag="data", loader_name="train_loader",
    batch_size=16, shuffle=True, is_training=True,
    compute_hash=False, preload_labels=True,
    preload_metadata=False, enable_h5_persistence=True,
)
test_loader = wl.watch_or_edit(
    test_ds, flag="data", loader_name="test_loader",
    batch_size=128, shuffle=False, is_training=False,
    compute_hash=False, preload_labels=True,
    preload_metadata=False, enable_h5_persistence=True,
)

# Watched losses + metric (they log themselves per sample)
train_criterion = wl.watch_or_edit(nn.CrossEntropyLoss(reduction="none"),
                                   flag="loss", signal_name="train-loss-CE", log=True)
test_criterion = wl.watch_or_edit(nn.CrossEntropyLoss(reduction="none"),
                                  flag="loss", signal_name="test-loss-CE", log=True)
metric = wl.watch_or_edit(Accuracy(task="multiclass", num_classes=10).to(device),
                          flag="metric", signal_name="metric-ACC", log=True)

## 5. Train & evaluate steps

The `guard_training_context` / `guard_testing_context` blocks tell WeightsLab which phase it's in. `criterion(..., batch_ids=ids, preds=preds)` passes the sample ids so the loss is stored **per sample**, and `wl.save_signals(...)` logs a custom per-sample accuracy signal during evaluation.

In [ ]:
def train(loader, model, optimizer, criterion, device):
    with guard_training_context:
        inputs, ids, labels = next(loader)
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(inputs)
        preds = logits.argmax(dim=1, keepdim=True)

        loss = criterion(logits.float(), labels.long(), batch_ids=ids, preds=preds)
        total_loss = loss.mean()
        total_loss.backward()
        optimizer.step()
    return total_loss.detach().cpu().item()


def test(loader, model, criterion, metric, device, n_batches):
    losses = torch.tensor(0.0, device=device)
    for inputs, ids, labels in loader:
        with guard_testing_context:
            inputs, labels = inputs.to(device), labels.to(device)
            logits = model(inputs)
            preds = logits.argmax(dim=1, keepdim=True)

            loss = criterion(logits, labels, batch_ids=ids, preds=preds)
            losses += loss.mean()
            metric.update(logits, labels)

            correct = (preds.view(-1) == labels.view(-1)).float()
            wl.save_signals(
                preds_raw=logits, targets=labels, batch_ids=ids, preds=preds,
                signals={
                    "test_metric/Accuracy_per_sample": correct,
                    "test_metric/Inverse_Accuracy_per_sample": 1.0 - correct,
                },
            )
    return (losses / n_batches).item(), (metric.compute() * 100).item()

## 6. (Optional) Expose the backend for the live UI

Skip this section if you only want the inline results above. To watch training **live in Weights Studio on your own machine**, set `EXPOSE_UI = True` below.

When it's on, `wl.serve(..., serving_bore=True)` (next section) downloads [`bore`](https://github.com/ekzhang/bore) and opens a **raw-TCP** tunnel over its free public relay (`bore.pub`) — **no signup, no account, no credit card** — then prints the exact `weightslab tunnel bore.pub:<port>` command to run locally.

> ⚠️ Raw TCP is required so gRPC's HTTP/2 frames pass through untouched (this is also why we avoid `ngrok`, which now needs a credit card for TCP endpoints).
>
> 🔓 `bore.pub` is a shared public relay: the random port is the only thing protecting your endpoint. Fine for a demo — don't stream sensitive data through it.

In [ ]:
EXPOSE_UI = True   # set to False for an inline-only run (no tunnel)

## 7. Serve & train

`wl.serve(serving_grpc=True)` starts the background gRPC server (non-blocking) that Weights Studio connects to. With `serving_bore=EXPOSE_UI` it also opens the `bore` tunnel and **prints the `weightslab tunnel bore.pub:<port>` command** to run on your machine. `wl.start_training(...)` flips the experiment into the *training* state, then we run the loop, periodically evaluating and exporting signals.

Leave this cell **running** while you watch it stream in the UI.

In [ ]:
wl.serve(serving_grpc=parameters["serving_grpc"], serving_bore=EXPOSE_UI)
wl.start_training(timeout=3)

steps = parameters["training_steps_to_do"]
eval_ratio = parameters["eval_full_to_train_steps_ratio"]
export_ratio = parameters["write_export_ratio"]
n_test_batches = len(test_loader)

test_loss = test_acc = None
pbar = tqdm(range(steps), desc="Training")
for step in pbar:
    age = model.get_age() if hasattr(model, "get_age") else step

    train_loss = train(train_loader, model, optimizer, train_criterion, device)

    if age > 0 and age % eval_ratio == 0:
        test_loss, test_acc = test(test_loader, model, test_criterion, metric, device, n_test_batches)

    if age > 0 and age % export_ratio == 0:
        wl.write_history()
        wl.write_dataframe()

    postfix = {"loss": f"{train_loss:.3f}"}
    if test_acc is not None:
        postfix["test_acc"] = f"{test_acc:.1f}%"
    pbar.set_postfix(postfix)

# Final export of signal history + per-sample dataframe
wl.write_history()
wl.write_dataframe()
print("Training complete. Logs at:", log_dir)

## 8. Which samples hurt the most?

WeightsLab exports a per-sample dataframe to `root_log_dir`. Ranking by loss surfaces the samples the model struggles with — the usual suspects for **mislabels and outliers**. This is the same signal the Studio UI visualizes; here we peek at it inline.

In [ ]:
import glob
import pandas as pd

csvs = sorted(glob.glob(os.path.join(log_dir, "**", "*.csv"), recursive=True),
              key=os.path.getmtime)
if csvs:
    df = pd.read_csv(csvs[-1])
    print(f"Loaded {csvs[-1]}  (shape={df.shape})")
    loss_cols = [c for c in df.columns if "loss" in c.lower()]
    if loss_cols:
        display(df.sort_values(loss_cols[-1], ascending=False).head(10))
    else:
        display(df.head())
else:
    print("No dataframe export found — run the training cell first.")

## 9. See it live in Weights Studio

Everything above ran headless. The real payoff is the **Weights Studio** UI, where you browse the highest-loss images, filter by class, and curate the dataset mid-training.

Studio runs as a local Docker stack (a static frontend + an Envoy gRPC-Web proxy). **Colab has no Docker daemon**, so you don't run the UI *inside* Colab — you run Studio on your own machine and point it at this notebook's backend using the `bore.pub:<port>` endpoint **printed when you ran Section 7** (`serving_bore=True`).

**On your machine** (with Docker Desktop):
```bash
pip install weightslab

# Terminal 1 — launch the UI (plaintext HTTP, the default)
weightslab ui launch                       # opens http://localhost:5173

# Terminal 2 — bridge the Colab backend to localhost:50051
weightslab tunnel bore.pub:12345           # <-- the host:port printed in Section 7
```

Then open **http://localhost:5173** — Studio connects through your local Envoy → `weightslab tunnel` → `bore.pub` → this Colab backend, and you watch training stream live.

> ℹ️ How it fits together: the UI's Envoy proxy already dials `localhost:50051`; `weightslab tunnel` simply makes the remote Colab backend appear there. It must stay **plaintext** end-to-end (the default `weightslab ui launch`, raw-TCP `bore`) so gRPC's HTTP/2 frames pass through untouched.

Prefer to keep it all local? Run this same example on your own machine (`weightslab start example --cls`) and launch the UI next to it — no tunnel required.

---

<div align="center">
Crafted with 💙 by <a href="https://github.com/GrayboxTech/weightslab">GrayboxTech</a> — if WeightsLab helps you catch a bad label, drop us a ⭐ on <a href="https://github.com/GrayboxTech/weightslab">GitHub</a>.
</div>